# Merging CHIRPS data and climate model data for comparison

In [1]:
# import necessary packages
import numpy as np
import pandas as pd
import dask.dataframe as dd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# mount google drive if needed
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Set index to be reindex to
new_lat = np.arange(-90, 90, 0.5)
new_lon = np.arange(-180, 180, 0.5)

In [4]:
# define dictionary of region coordinates
region_coords = {'south_sudan': {'latitude': (3.5, 12.5),
                                 'longitude': (25, 35)},
                 'eastern_east_africa':  {'latitude': (-3.5, 8),
                                 'longitude': (38, 50)}}

In [21]:
# load in climate models
CanESM5 = xr.open_mfdataset('/content/drive/My Drive/capstone_data/NMME/CanESM5/prec/*2003*.nc', parallel=True)
CMCC = xr.open_mfdataset('/content/drive/My Drive/capstone_data/CDS/CMCC/prec/*2003*.nc', parallel=True)

# reindex and assign coordinates for each climate model
CanESM5 = CanESM5.assign_coords(X=(((CanESM5.X + 180) % 360) - 180)).sortby(['X'])
CanESM5 = CanESM5.reindex(X=new_lon, Y=new_lat).ffill('X').ffill('Y')

CMCC = CMCC.assign_coords(X=(((CMCC.X + 180) % 360) - 180)).sortby(['X'])
CMCC = CMCC.reindex(X=new_lon, Y=new_lat).ffill('X').ffill('Y')

In [9]:
# load in chirps
chirps = xr.open_dataset('/content/drive/My Drive/capstone_data/CHIRPS/chirps-v2.0.monthly.nc')

In [25]:
# define dictionary of opened models
models = {'CanESM5': CanESM5,
          'CMCC': CMCC}

In [26]:
# for every model, subset a region, merge it with CHIRPS, and save the netcdf file
def merge_chirps_and_model(model_dict, region_coord_dict, chirps, save_path):
    """This function takes a dictionary of models, a dictionary of region coordinates, CHIRPS data, and a save path.
    It subsets each model for each region, merges the subsetted model with CHIRPS, and saves the merged data as a netCDF file.
    """

    for model_name, model_data in model_dict.items():
        for region_name, coord_dict in region_coord_dict.items():
            print(f'Merging {model_name} and CHIRPS for {region_name}')

            # Subset a region for the current model
            current_model_region = (model_data.sel(Y=slice(coord_dict['latitude'][0], coord_dict['latitude'][1]), X=slice(coord_dict['longitude'][0], coord_dict['longitude'][1]))
                                .rename(
                {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

            # Interpolate and subset CHIRPS to match spatial resolution of NMME
            # must add 0.5 degrees to each lat/long value to chirps subset for interp_like to work
            # The longitude slice was incorrect, resulting in an empty array
            # Changed the end value of the longitude slice to coord_dict['longitude'][1] + 0.5
            chirps_region = chirps.sel(latitude=slice(coord_dict['latitude'][0] + 0.5, coord_dict['latitude'][1] + 0.5),
                                            longitude=slice(coord_dict['longitude'][0] + 0.5, coord_dict['longitude'][1] + 0.5)).interp_like(current_model_region,
                                                                                                            method='nearest')
            # Calculate realized dates for the current model and region
            current_model_region_df = current_model_region.to_dataframe().reset_index()
            current_model_region_df['realization time'] = current_model_region_df['date of prediction'] + (
                        current_model_region_df['lead_time'] * 30).astype('timedelta64[D]')
            current_model_region_df['month'] = current_model_region_df['realization time'].dt.month
            current_model_region_df['year'] = current_model_region_df['realization time'].dt.year

            # Convert CHIRPS to dataframe
            chirps_region_df = chirps_region.to_dataframe().reset_index()
            chirps_region_df['month'] = chirps_region_df['time'].dt.month
            chirps_region_df['year'] = chirps_region_df['time'].dt.year

            # Merge CHIRPS and current_model_region dataframe
            current_model_region_merged_df = current_model_region_df.merge(chirps_region_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

            print(current_model_region_merged_df.head())

            model_region_save_path = f'{save_path}{region_name}_{model_name}_merged.nc'

            # Save to netCDF
            # current_model_region_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf(model_region_save_path)

In [27]:
save_path = 'data/netCDF/merged/'
merge_chirps_and_model(models, region_coords, chirps, save_path)

Merging CanESM5 and CHIRPS for south_sudan
   lead_time  latitude    M date of prediction  longitude  predicted_precip  \
0        0.5       3.5  1.0         2003-01-01       25.0          2.324761   
1        0.5       3.5  1.0         2003-01-01       25.5          2.324761   
2        0.5       3.5  1.0         2003-01-01       26.0          2.202807   
3        0.5       3.5  1.0         2003-01-01       26.5          2.202807   
4        0.5       3.5  1.0         2003-01-01       27.0          1.963911   

  realization time  month  year       time  precip  
0       2003-01-16      1  2003 2003-01-01     NaN  
1       2003-01-16      1  2003 2003-01-01     NaN  
2       2003-01-16      1  2003 2003-01-01     NaN  
3       2003-01-16      1  2003 2003-01-01     NaN  
4       2003-01-16      1  2003 2003-01-01     NaN  
Merging CanESM5 and CHIRPS for eastern_east_africa
   lead_time  latitude    M date of prediction  longitude  predicted_precip  \
0        0.5      -3.5  1.0       